# Final runs (v2.0): class-aware grounding (RQ1), ablations, plan quality and statistics
Attach the output of the last notebook that ran step 04 (it holds the models, teacher cache and earlier runs). Run with **Save Version -> Save & Run All** (about 6-9 h).

In [ ]:
# === STEP 0: SETUP — run this cell first, every time =========================
# Before running: in the right-hand panel
#   * "Session options" -> Accelerator = GPU T4 x2, Internet = On
#   * "Add Input" -> your code dataset (the uploaded bcd-project.zip)
#   * "Add Input" -> search "brain tumor mri dataset" by masoudnickparvar
#   * (notebooks 02-05) "Add Input" -> "Your Work" -> the previous notebook(s)
import os, sys, glob, shutil, subprocess

def find_dir(test, root='/kaggle/input', max_depth=6):
    # walk the attached inputs and return the first folder that passes test(folder)
    root_depth = root.rstrip('/').count('/')
    for d, subdirs, files in os.walk(root):
        if d.count('/') - root_depth > max_depth:
            subdirs[:] = []
            continue
        if test(d, subdirs, files):
            return d
    return None

# 1) the project code: the folder that contains bcd/__init__.py
code_dir = find_dir(lambda d, s, f: os.path.isfile(os.path.join(d, 'bcd', '__init__.py')))
if code_dir is None:
    raise SystemExit('Code not found. Click "Add Input" and attach the dataset you created from bcd-project.zip.')
shutil.copytree(code_dir, '/kaggle/working/bcd-project', dirs_exist_ok=True)
sys.path.insert(0, '/kaggle/working/bcd-project')
print('code found at:', code_dir)

# 2) install the extra libraries (takes ~1-2 minutes)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.45,<4.52', 'peft>=0.12',
                'bitsandbytes>=0.43', 'accelerate>=0.33', 'ultralytics', 'imagehash', 'h5py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])  # old torchao on Kaggle breaks peft

# 3) the brain MRI dataset: the folder that has Training/ and Testing/ inside
from bcd.config import load_config
cfg = load_config('/kaggle/working/bcd-project/configs/default.yaml')
mri = find_dir(lambda d, s, f: 'Training' in s and 'Testing' in s)
if mri is None:
    raise SystemExit('MRI dataset not found. Click "Add Input" and attach masoudnickparvar/brain-tumor-mri-dataset.')
cfg.paths.kaggle_root = mri
print('MRI dataset found at:', mri)

# 4) results of earlier notebooks (a folder named bcd that contains manifest.csv)
os.makedirs(cfg.paths.work_dir, exist_ok=True)
for prev in glob.glob('/kaggle/input/**/bcd/manifest.csv', recursive=True):
    src = os.path.dirname(prev)
    if not src.startswith('/kaggle/working'):
        shutil.copytree(src, cfg.paths.work_dir, dirs_exist_ok=True)
        print('restored earlier results from:', src)
print('GPUs available:', __import__('torch').cuda.device_count())
print('Setup complete. Work folder:', cfg.paths.work_dir)


In [ ]:
import glob, json, os
from bcd.runtime import build_runtime, run_condition, gate_study_v2, results_dump, plan_quality, adapter_state_from_dir, significance, pack_results, work
W = cfg.paths.work_dir
yolo = glob.glob(f'{W}/yolo_runs/tumour/weights/best.pt')[0]
rt = build_runtime(cfg, f'{W}/adapters/student_seed', f'{W}/adapters/teacher', yolo)
print(len(rt['stream']), 'stream cases;', len(rt['train_boxes']), 'training images with YOLO boxes for the location model')

In [ ]:
# 1) RQ1 on validation: original vs class-aware grounding terms, weights chosen by 5-fold cross-validation
gs = gate_study_v2(cfg, rt)

In [ ]:
# 2) proposed system with the new gate + cumulative online loop; then the ablations of each grounding term
import traceback
def run(cond, seed=0):
    if os.path.exists(f'{W}/runs/{cond}_seed{seed}/summary.json'):
        print('skip', cond, seed); return json.load(open(f'{W}/runs/{cond}_seed{seed}/summary.json'))
    try:
        s = run_condition(cfg, rt, cond, seed)
        print(cond, seed, s['gate'], s['final_student_test']); return s
    except Exception:
        traceback.print_exc(); print('RUN FAILED:', cond, seed); return None
sP = run('P_v3', 0)
w = (sP or {}).get('gate', {})
print('P_v3 gate weights selected on validation:', w)
if w.get('w2', 0) > 0: run('A1_v3', 0)      # without A_YOLO: shows what the YOLO term contributes
if w.get('w3', 0) > 0: run('A2_v3', 0)      # without A_XAI: shows what the Grad-CAM term contributes
if w.get('w2', 0) == 0 and w.get('w3', 0) == 0: run('P_v3', 1)   # no grounding weight: repeat run instead

In [ ]:
# 3) treatment-plan quality (Appendix A.2 judge): seed student vs final student vs teacher, 30 test tumour images
from bcd.runtime import best_run
br = best_run(cfg)
print('final student from run:', br)
try:
    plan_quality(cfg, rt, {'seed student': rt['student'].seed_state, f'final student ({br})': adapter_state_from_dir(f'{W}/runs/{br}/adapter')}, n=30)
except Exception:
    traceback.print_exc(); print('PLAN QUALITY FAILED - the other results are kept')

In [ ]:
# 4) statistics: CIs for every run and paired tests (same test images / same stream cases)
pairs = [('P_v3_seed0', 'P_v2_seed0'), ('P_v3_seed0', 'P_seed0'), ('P_v2_seed0', 'P_seed0'), ('P_v3_seed0', 'B2_seed0'),
         ('P_v2_seed0', 'B5_v2_seed0'), ('P_v2_seed0', 'B3_v2_seed0'), ('P_v3_seed0', 'A1_v3_seed0'), ('P_v3_seed0', 'A2_v3_seed0'),
         ('P_v3_seed0', 'P_v3_seed1'), ('P_v2_seed0', 'B4_seed0')]
significance(cfg, rt, pairs)
results_dump(cfg)
for f in ('gate_study_v2.json', 'table_gate_study_v2.csv', 'table_plan_quality.csv'):
    if os.path.exists(f'{W}/{f}'): print('==', f); print(open(f'{W}/{f}').read())
print('download:', pack_results(cfg))